# 06: Real-World Holdout Validation & Agentune AutoML Overfitting Guardrails

### Empirical Evidence on Production Datasets and Automated Feature Generation Pipelines

---

### Executive Overview & Motivation

In previous notebooks, we demonstrated that unassisted LLM exploratory agents suffer from **Search Space Amnesia**, running hundreds of exploratory hypothesis evaluations while reporting only their top-scoring findings. This inflates the False Discovery Rate (FDR) up to 80–90% on synthetic control datasets (`null_honeypot`, `needle_in_haystack`, `correlated_blocks`).

However, a natural skepticism arises:
> *"Were those synthetic benchmarks specifically engineered to highlight SignalGate's theoretical edge? How does SignalGate perform on real-world production datasets? And what happens when an automated feature discovery engine—like SparkBeyond Agentune—generates thousands of candidate interaction features?"*

In this notebook, we address both questions with rigorous empirical experiments:

1. **Part 1: Real-World Production Datasets with Frontier LLMs**
   - **Lending Club Loan Defaults**: 54,655 historical loans, 42 features, predicting binary default (`bad_loans`).
   - **King County House Sales**: 21,610 home transactions, predicting premium valuations (`high_value`).
   - True **Out-of-Sample (OOS) Holdout Partitioning**: Models explore strictly on $N_{\text{train}} = 5,000$, and promoted drivers are evaluated on an unseen holdout vault of $N_{\text{test}} = 2,500$.
   - Live evaluations with **Google Gemini 3.5 Flash Lite** across three agent archetypes:
     - `business_analyst` (unassisted greedy search)
     - `statistician_unassisted` (prompt-only statistical guidance)
     - `statistician_signalgate` (governed by stateful MCP noise ceiling ledger)

2. **Part 2: SparkBeyond Agentune AutoML Overfitting Benchmark**
   - Automated feature engineering engines (e.g., [SparkBeyond Agentune](https://github.com/SparkBeyond/agentune)) synthesize high-dimensional spaces of candidate interaction features: pairwise boolean conjunctions ($A \land B$), disjunctions ($A \lor B$), subgroup slices, and non-linear ratios.
   - When generating $M = 1,000+$ features, greedy selection on training data suffers from **Extreme Value Theory (EVT)** noise capture: random interactions achieve spuriously high training scores.
   - We benchmark **Unassisted Agentune** vs. **SignalGate-Governed Selection** on holdout replication rate, out-of-sample decay, and downstream classifier generalization (ROC AUC).


---
## Part 1: Real-World Empirical Holdout Audit with Frontier LLMs

We benchmark three agent archetypes exploring real-world production data under strict holdout partitioning:
- Exploration is restricted to `df_train` ($N=5,000$).
- Evaluation is conducted strictly on `df_test` ($N=2,500$) using non-parametric permutation tests, Chi-square/Fisher exact tests, Relative Information Gain (RIG), and Generalization Bias:
$$\text{Generalization Bias} = \text{RIG}_{\text{train}} - \text{RIG}_{\text{holdout}}$$
$$\text{Decay \%} = \max\left(0, 1 - \frac{\text{RIG}_{\text{holdout}}}{\text{RIG}_{\text{train}}}\right) \times 100\%$$


In [1]:
import pandas as pd
from IPython.display import display

# Load real-world benchmark summary tables
lc_summary = pd.read_csv('../Data/real_world_eval_lending_club_gemini-3.5-flash-lite_summary.csv')
kc_summary = pd.read_csv('../Data/real_world_eval_king_county_gemini-3.5-flash-lite_summary.csv')

print("=== LENDING CLUB LOAN DEFAULTS (Frontier LLM Holdout Audit) ===")
display(lc_summary)

print("\n=== KING COUNTY HOUSE SALES (Frontier LLM Holdout Audit) ===")
display(kc_summary)


=== LENDING CLUB LOAN DEFAULTS (Frontier LLM Holdout Audit) ===
     Dataset                 Model                 Persona  Holdout_Replication_Rate (%)  Real_FDR (%)  Mean_Train_RIG  Mean_Holdout_RIG  Generalization_Bias  Decay (%)  Under_Reporting_Ratio
lending_club gemini-3.5-flash-lite        business_analyst                     33.333333     66.666667        0.015890          0.015786             0.000103  68.906667               8.229630
lending_club gemini-3.5-flash-lite statistician_signalgate                    100.000000      0.000000        0.084132          0.095707            -0.011576   1.090476               1.197531
lending_club gemini-3.5-flash-lite statistician_unassisted                    100.000000      0.000000        0.051387          0.048761             0.002624   6.948333               6.658201

=== KING COUNTY HOUSE SALES (Frontier LLM Holdout Audit) ===
    Dataset                 Model                 Persona  Holdout_Replication_Rate (%)  Real_FDR (%)  Mea

### Figure 4: Holdout Replication Rate and Search Space Amnesia

The figure below visualizes the empirical holdout audit across both real-world production datasets.


In [1]:
from IPython.display import Image, display
display(Image(filename='figures/fig4_real_world_frontier_replication.png'))


### Key Real-World Findings:

1. **Unassisted Business Analysts Suffer 66.7% Real FDR on Real Data**:
   - The unassisted `business_analyst` tested between 35 and 45 candidate features and subgroup slices during its exploratory steps, but presented only 4–5 "compelling findings" in its final executive summary.
   - Because it failed to account for the $\approx 8\times$ multiple testing penalty, **66.7% of its promoted discoveries failed to replicate out-of-sample** ($p > 0.05$ on `df_test`), with an average holdout decay of **68.9%**.
2. **Prompt-Only Statistician Is Under-Reporting and Overly Timid**:
   - The unassisted statistician sticks strictly to obvious baseline features (`grade`, `sqft_living_bin`), achieving 100% replication but suffering severe under-reporting (4.5–6.6x) and low exploratory power (finding only simple univariates).
3. **SignalGate Achieves Stateful Ledger Precision & Higher Power**:
   - The `statistician_signalgate` maintained exact state tracking (Under-Reporting Ratio = $1.00\times$ on King County, $1.20\times$ on Lending Club).
   - Because it safely evaluated compound signals against closed-form EVT thresholds $\tau_{\text{sig}}(M)$, it discovered genuine multi-column drivers that generalized out-of-sample, achieving **nearly $2\times$ higher Holdout RIG** (0.0957 vs 0.0488 on Lending Club) with **0.0% False Discovery Rate**.


---
## Part 2: SparkBeyond Agentune AutoML Overfitting Benchmark

### The Mechanics of Feature Generation in Agentic AutoML

Tools like [SparkBeyond Agentune](https://github.com/SparkBeyond/agentune) take a tabular dataset and programmatically synthesize thousands of feature candidates:
1. **Univariate Category Bins**: $I(X_i = v)$
2. **Pairwise Conjunctions (AND)**: $I(X_i = v_a) \land I(X_j = v_b)$
3. **Pairwise Disjunctions (OR)**: $I(X_i = v_a) \lor I(X_j = v_b)$
4. **Subgroup Slice Filters & Ratios**

### The Multiple-Testing Trap in Greedy AutoML
When evaluating $M = 1,000$ synthetic interaction candidates on a finite training split ($N = 5,000$):
- Pure noise features and small-cell subgroups inevitably achieve observed Relative Information Gain of $\text{RIG} \approx 0.03 - 0.06$ purely by chance (**Winner's Curse / Extreme Value Theory**).
- **Unassisted Agentune** greedily ranks all $M$ candidates by training score and selects the top $K$ features.
- **SignalGate-Governed Selection** tracks testing volume $M$ inside the `NoiseGateSessionLedger`, dynamically raising the significance ceiling:
$$\tau_{\text{sig}}(M, \alpha) = \frac{\chi^2_{1, 1 - \alpha/M}}{2 N \ln 2}$$
and enforces minimum contingency cell size constraints to reject small-cell overfitting artifacts.


In [1]:
# Load Agentune benchmark results on Lending Club
agentune_summary = pd.read_csv('../Data/agentune_benchmark_lending_club_summary.csv')
agentune_downstream = pd.read_csv('../Data/agentune_benchmark_lending_club_downstream.csv')

print("=== AGENTUNE AUTOML FEATURE SELECTION BENCHMARK (M = 1000 Candidates) ===")
display(agentune_summary)

print("\n=== DOWNSTREAM ML CLASSIFIER GENERALIZATION (Overfitting Gap) ===")
display(agentune_downstream)


=== AGENTUNE AUTOML FEATURE SELECTION BENCHMARK (M = 1000 Candidates) ===
                           Approach  Total_Generated_M  Selected_Features  Holdout_Replication_Rate (%)  Real_FDR (%)  Mean_Train_RIG  Mean_Holdout_RIG  Generalization_Bias  Decay (%)
Unassisted Agentune (Greedy AutoML)               1000                 15                     73.333333     26.666667        0.035556          0.022455             0.013101  39.195033
      SignalGate-Governed Selection               1000                 15                    100.000000      0.000000        0.031442          0.026031             0.005410  21.887352

=== DOWNSTREAM ML CLASSIFIER GENERALIZATION (Overfitting Gap) ===
                            Pipeline  N_Features  Train_AUC  Holdout_Test_AUC  Overfitting_Gap
               Baseline Raw Features         119   0.729152          0.700973         0.028179
Baseline + Unassisted Agentune Top-K         134   0.730266          0.699346         0.030920
Baseline + SignalGate-

### Figure 5: Feature Holdout Decay Curves

Comparing the train vs holdout performance of individual interaction features selected by Unassisted Agentune vs SignalGate.


In [1]:
display(Image(filename='figures/fig5_agentune_holdout_decay.png'))


### Figure 6: The Downstream AutoML Overfitting Gap

Evaluating downstream model performance (ROC AUC) on pure holdout test data. Notice how unassisted AutoML increases the overfitting gap while lowering holdout performance, whereas SignalGate achieves genuine holdout lift.


In [1]:
display(Image(filename='figures/fig6_agentune_downstream_overfitting_gap.png'))


---
## Quantitative Insights & Synthesis

| Benchmark Dimension | Unassisted Agentune (Greedy AutoML) | SignalGate-Governed Selection | SignalGate Advantage |
| :--- | :---: | :---: | :---: |
| **Search Space Accounting** | Ignored ($M=1$ assumption) | Stateful Ledger ($M=1,000$) | Eliminates Selection Bias |
| **Holdout Replication Rate** | **73.3%** | **100.0%** | **+26.7% higher replication** |
| **Real False Discovery Rate (FDR)** | **26.7%** | **0.0%** | **Zero spurious drivers** |
| **Out-of-Sample Decay %** | **39.2%** | **21.9%** | **1.8x lower decay** |
| **Downstream Holdout AUC** | **0.6993** (Degraded vs baseline) | **0.7029** (Superior to baseline) | **Real generalization lift** |
| **Downstream Overfitting Gap** | **+0.0309** (Widened) | **+0.0274** (Narrowed) | **No interaction overfitting** |

### Why Unassisted Agentic AutoML Degrades in Production:
1. **The Spurious Conjunction Trap**:
   When combining 15 binned features into pairs, there are $\approx 3,000$ possible conjunctions. In any finite training sample ($N=5,000$), arbitrary pairs (e.g. `(home_ownership == 'RENT') & (purpose == 'wedding')`) will show high observed default correlations simply because the contingency cell is small and noisy.
2. **The Winner's Curse in Forward Feature Selection**:
   By greedily selecting the top-scoring interactions without testing-budget adjustments, unassisted AutoML loads downstream models with noise coefficients.
3. **The SignalGate Remedy**:
   SignalGate computes the exact closed-form threshold $\tau_{\text{sig}}(M)$ required for a feature to be distinguished from the maximum of $M$ independent noise trials, coupled with sample-size cell guards. Only features that reflect underlying structural truth survive the gate.

---
## Conclusion

This evaluation resolves both concerns:
1. **Real-world production datasets** (Lending Club and King County) validate that unassisted LLM exploratory agents suffer severe false discoveries (FDR up to 66.7%) due to search-space amnesia, while SignalGate maintains $1.0\times$ ledger discipline and cuts false discoveries to zero.
2. **Agentic AutoML workflows** (such as SparkBeyond Agentune) generate severe feature overfitting when optimizing interactions greedily. SignalGate provides the exact statistical guardrails necessary to transform autonomous feature synthesis into robust, replicable production pipelines.
